In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from YOLO_loader import BoundingBox, CustomImage
import torch

In [ ]:
class_to_id = {
    "card":1,
    "screen":2
}

N_CELL = 52
IMG_SIZE = 416
N_CLASS = 2
N_ANCHORS = 3
TARGET_SIZE = (IMG_SIZE,IMG_SIZE)
H = 416
W = 416
S = 52  
C = len(class_to_id)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
DIRECTORY = "dataset_yolo_format"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
from YOLO_loader import get_labels


labels = get_labels(DIRECTORY)
ratios = []
for label in labels:
    boxes = label.get_bounding_boxes()
    for box in boxes :
        vec_ratio = box.get_ratio()
        ratios.append(vec_ratio)

ratios = np.array(ratios)

k = 3
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(ratios)  

labels = kmeans.labels_
centers = kmeans.cluster_centers_


plt.scatter(ratios[:, 0], ratios[:, 1], c=labels, cmap='viridis', s=30)
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='x', s=100)
plt.title("Clusters K-Means")
plt.show()

In [ ]:
from dataset import CardRecognitionDataset, custom_collate_fn
from torch.utils.data import Subset,DataLoader

dataset = CardRecognitionDataset(directory= DIRECTORY, n_cells=N_CELL,is_dark_and_white=False,target_size=TARGET_SIZE,bounding_boxes_ratio=centers)

STD = dataset.get_std()
MEAN = dataset.get_mean()

N = dataset.__len__()
train_size = int(0.7 * N)
val_size   = int(0.2 * N)   
indices = torch.randperm(N)
test_size  = N - train_size - val_size

train_indices = indices[:train_size]
val_indices   = indices[train_size:train_size+val_size]
test_indices  = indices[train_size+val_size:]


train_dataset = Subset(dataset, train_indices)
val_dataset   = Subset(dataset, val_indices)
test_dataset  = Subset(dataset, test_indices)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,collate_fn=custom_collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)


In [ ]:
import math
for custom_imgs, images, labels in train_loader:

    # Récupération des images annotées
    image_lst = [custom_img.get_image() for custom_img in custom_imgs]
    images_np = np.array(image_lst)  # (B,H,W,C)

    batch_size = images_np.shape[0]
    grid_size = math.ceil(math.sqrt(batch_size))  # Taille de la grille

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(4*grid_size, 4*grid_size))
    axes = axes.flatten()  # Pour itérer facilement même si c'est 2D

    for i in range(batch_size):
        axes[i].imshow(images_np[i])
        axes[i].axis('off')

    # Masquer les axes vides si batch_size < grid_size**2
    for i in range(batch_size, grid_size**2):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()